# Duplicate Data Handling

Duplicate records are rows that represent the same observation more than once.

Duplicates can occur because of repeated data entry, file merging, system errors, or repeated data collection.

In this notebook, we will identify and analyze duplicate records in the Titanic dataset before deciding whether they should be removed.

The main goal is not to delete duplicates blindly. We first understand why the duplicates exist and then apply an appropriate business rule.

In [2]:
import pandas as pd

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## What are Duplicate Records?

A duplicate record is a row that contains the same information as another row.

For example, if the same passenger record appears twice in a dataset, it may be an exact duplicate.

Duplicates can affect analysis because the same observation may receive more importance than it should.

### Example

If one passenger is accidentally recorded twice, counting both records may incorrectly increase the number of passengers and may also affect statistical analysis.

Therefore, duplicate records should be identified before further preprocessing.

In [3]:
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate records:", duplicate_count)

Number of exact duplicate records: 0


## Exact Duplicates

Exact duplicates are records where all column values are exactly the same.

### Problem

The same complete record may appear more than once.

### Analysis

Exact duplicates may be caused by repeated data entry or combining the same data multiple times.

### Technique Selected

We use `duplicated()` to identify exact duplicate rows.

### Reason

This method compares the complete row and provides a simple way to detect repeated records.

### Implementation

We will identify the duplicate rows without deleting them first.

### Result

The output shows how many complete duplicate records are present.

### Impact

If true duplicate observations are kept, they can give certain observations more weight and may affect statistical analysis and Machine Learning models.

In [4]:
exact_duplicates = df[df.duplicated(keep=False)]

print("Exact duplicate records:")
print(exact_duplicates)

print("\nNumber of exact duplicate records:", len(exact_duplicates))

Exact duplicate records:
Empty DataFrame
Columns: [PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked]
Index: []

Number of exact duplicate records: 0


## Partial Duplicates

A partial duplicate occurs when records are not completely identical but share the same values in important columns.

For example, two Titanic records may have the same passenger name or ticket number while other columns are different.

### Problem

A record may look like a duplicate based on selected business fields but may actually represent a different observation.

### Analysis

Partial duplicates require more careful investigation than exact duplicates because matching one or two columns does not automatically mean that the records are duplicates.

### Technique Selected

We check duplicate values using selected columns.

### Reason

Selected-column duplicate detection helps identify possible repeated entities while keeping the original records available for investigation.

### Result

The output shows records that share the same value in the selected column.

### Impact

Removing valid records based only on one matching column could cause useful information to be lost.

In [5]:
partial_duplicates = df[df.duplicated(subset=["Name"], keep=False)]

print("Number of records with repeated passenger names:", len(partial_duplicates))

partial_duplicates[["Name", "Sex", "Age", "Ticket", "Fare"]].head(10)

Number of records with repeated passenger names: 0


,Name,Sex,Age,Ticket,Fare


## Identifying Duplicates Using `duplicated()`

The Pandas `duplicated()` function is used to identify records that have already appeared in the dataset.

By default, it checks the complete row.

We can also use the `subset` parameter to check duplicates based on specific columns.

This allows us to investigate both exact duplicates and possible partial duplicates before making a preprocessing decision.

In [6]:
print("Exact duplicate flags:")
print(df.duplicated().value_counts())

print("\nDuplicate flags based on Ticket:")
print(df.duplicated(subset=["Ticket"], keep=False).value_counts())

Exact duplicate flags:
False    891
Name: count, dtype: int64

Duplicate flags based on Ticket:
False    547
True     344
Name: count, dtype: int64


## Duplicate Detection Based on Selected Columns

Sometimes a dataset contains an identifier or business field that can be used to investigate repeated records.

In the Titanic dataset, `Ticket` can be useful for investigation because multiple passengers may legitimately share the same ticket.

### Problem

The same ticket number can occur for multiple passengers.

### Analysis

A repeated ticket number does not necessarily mean that the records are duplicates.

### Technique Selected

We identify repeated values in the `Ticket` column.

### Reason

This helps us understand the data structure without incorrectly deleting valid passenger records.

### Result

The output shows ticket values associated with multiple records.

### Impact

This prevents us from treating legitimate group information as duplicate data.

In [7]:
ticket_duplicates = df[df.duplicated(subset=["Ticket"], keep=False)]

print("Records belonging to repeated ticket numbers:", len(ticket_duplicates))

ticket_duplicates[["Name", "Ticket", "Pclass", "Fare", "Embarked"]].head(10)

Records belonging to repeated ticket numbers: 344


,Name,Ticket,Pclass,Fare,Embarked
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",113803,1,53.1000,S
7,"Palsson, Master. Gosta Leonard",349909,3,21.0750,S
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",347742,3,11.1333,S
9,"Nasser, Mrs. Nicholas (Adele Achem)",237736,2,30.0708,C
10,"Sandstrom, Miss. Marguerite Rut",PP 9549,3,16.7000,S
13,"Andersson, Mr. Anders Johan",347082,3,31.2750,S
16,"Rice, Master. Eugene",382652,3,29.1250,Q
20,"Fynney, Mr. Joseph J",239865,2,26.0000,S
24,"Palsson, Miss. Torborg Danira",349909,3,21.0750,S
25,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",347077,3,31.3875,S


## Handling Duplicate Records

Duplicate records should only be removed after confirming that they represent the same observation.

### Problem

Keeping true duplicate rows can introduce repeated observations into the analysis.

### Analysis

Before removing duplicates, we need to determine whether they are accidental duplicates or valid repeated records.

### Technique Selected

For confirmed exact duplicates, we can use `drop_duplicates()`.

### Reason

`drop_duplicates()` removes repeated complete records while keeping one valid copy.

### Implementation

We create a separate cleaned DataFrame instead of modifying the original dataset directly.

### Result

The number of rows after duplicate removal can be compared with the original number of rows.

### Impact

Removing confirmed duplicates can improve data quality and prevent repeated observations from influencing Machine Learning models.

In [8]:
df_clean = df.drop_duplicates()

print("Original number of rows:", len(df))
print("Rows after removing exact duplicates:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original number of rows: 891
Rows after removing exact duplicates: 891
Rows removed: 0


## Business Rules for Duplicate Removal

Duplicate removal should be based on the meaning of the data, not only on a technical rule.

For the Titanic dataset, multiple passengers can legitimately have the same ticket, cabin, or other values.

Therefore, matching values in one column should not automatically be considered a duplicate.

A safe business rule is:

- Remove a record when the complete row is an accidental duplicate.
- Investigate partial duplicates before removing them.
- Do not remove records only because they share a ticket or another common value.
- Preserve valid observations that represent different passengers.

This approach reduces the risk of deleting useful information.

In [9]:
print("Original shape:", df.shape)
print("Shape after exact duplicate removal:", df_clean.shape)

print("\nDuplicate rows remaining:")
print(df_clean.duplicated().sum())

Original shape: (891, 12)
Shape after exact duplicate removal: (891, 12)

Duplicate rows remaining:
0


## Why Blindly Deleting Duplicates Can Be Dangerous

Not every repeated value represents a duplicate record.

For example, several Titanic passengers may have travelled using the same ticket. If we remove all records with repeated ticket numbers, we could accidentally remove valid passenger information.

Therefore, duplicate handling should consider the business meaning of each column.

### Insight

The important lesson is that duplicate detection and duplicate removal are two different steps. We should first identify possible duplicates, understand why they exist, and then decide whether removal is appropriate.

### AI/ML Relevance

Incorrect duplicate removal can lead to loss of useful training data and may change the distribution of the dataset.

On the other hand, keeping confirmed accidental duplicates can give repeated observations excessive influence during Machine Learning training.

Therefore, careful duplicate handling helps maintain reliable and representative training data.

In [10]:
print("Duplicate Data Summary")
print("----------------------")
print("Total records:", len(df))
print("Exact duplicate records:", df.duplicated().sum())
print("Records after exact duplicate removal:", len(df_clean))
print("Exact duplicates remaining:", df_clean.duplicated().sum())

Duplicate Data Summary
----------------------
Total records: 891
Exact duplicate records: 0
Records after exact duplicate removal: 891
Exact duplicates remaining: 0


## Final Preprocessing Decision

### Problem
The dataset was checked for exact and partial duplicate records.

### Analysis
Exact duplicates represent completely repeated rows, while repeated values in selected columns may represent valid relationships rather than duplicates.

### Technique Selected
`duplicated()` was used for detection and `drop_duplicates()` was used only for confirmed exact duplicates.

### Reason
This approach is safer than removing records based only on one column such as `Ticket` or `Name`.

### Implementation
Exact duplicate records were removed into a separate DataFrame named `df_clean`.

### Result
The original and cleaned dataset sizes were compared to determine how many exact duplicate records were removed.

### Impact
Removing confirmed duplicates helps prevent repeated observations from influencing statistical analysis and Machine Learning models, while careful investigation of partial duplicates helps prevent valid information from being lost.

### Conclusion
Duplicate handling should be based on both technical detection and the business meaning of the data. In this Titanic dataset, repeated ticket values are not automatically duplicates because multiple passengers can legitimately share the same ticket.